> **Fixed version note**
>
> - Added a safer path lookup for `chinook.db`.
> - Kept the teaching flow intact because the notebook already ran correctly.
> - This file was validated with the uploaded local database.

# Lecture 1: Jupyter Notebooks & SQL

## What is a Jupyter Notebook?
A Jupyter Notebook is an incredibly popular open-source web application that allows you to create and share documents that contain live code, equations, visualizations, and narrative text. Think of it as a digital lab notebook for programmers and data scientists. 

Instead of just writing raw code in a text file, you can write code in "chunks," see the results immediately right below that chunk, and explain your thought process with formatted text and images—all in the same document.

## Introduction to SQL

**SQL** stands for **Structured Query Language**. It is the language that lets you access and manipulate databases.

We use **Relational Database Management Systems (RDBMS)**:
* **Relational:** Data is stored in Tables that are linked to each other by unique keys (IDs).
* **Management System:** It is a software application (like SQL) that wraps around the data to manage who can see it and how fast it can be retrieved. 

### Why use SQL?
Real-world data often contains hundreds of thousands of rows. We can’t load them all to our computer because it is too heavy. We use SQL to pre-process the data in the database and get only the parts we care about.

## Setup: Python + SQL

We will use **Pandas** and **sqlite3** to connect to the **Chinook Database**. Pandas is the most important data library for this task. We will get very familiar with pandas in the upcoming lessons, for now we will install it for this lecture.

First we will install pandas:

### Install dependencies

In [102]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


We define a helper function `run_query()` to easily run SQL commands and return the results as a Pandas DataFrame.

### Import libraries

In [103]:
import pandas as pd
import sqlite3

In [104]:
from pathlib import Path
import pandas as pd
import sqlite3

# FIX: use a robust local path so the notebook works from the repo root
db_candidates = [Path('chinook.db'), Path('/mnt/data/chinook.db')]
db_path = next((p for p in db_candidates if p.exists()), None)
if db_path is None:
    raise FileNotFoundError("Could not find chinook.db in the working directory.")

db_filename = str(db_path)

def run_query(q):
    # This creates a connection, runs the query, and closes the connection automatically
    with sqlite3.connect(db_filename) as conn:
        return pd.read_sql(q, conn)

# Test the connection
try:
    print("Connecting to database...")
    tables = run_query("SELECT name FROM sqlite_master WHERE type='table';")
    print("Success! Tables found:")
    print(tables)
except Exception as e:
    print(f"Error: {e}")
    print("Double-check that 'chinook.db' is present and readable.")


Connecting to database...
Success! Tables found:
              name
0            album
1           artist
2         customer
3         employee
4            genre
5          invoice
6     invoice_line
7       media_type
8         playlist
9   playlist_track
10           track


### Database Tables
A database most often contains one or more tables. Each table is identified by a name (e.g., "album" or "artist") and contains records (rows) with data.

We can see the list of tables in our database below:

In [105]:
run_query("SELECT name FROM sqlite_master WHERE type='table';")

,name
0,album
1,artist
2,customer
3,employee
4,genre
5,invoice
6,invoice_line
7,media_type
8,playlist
9,playlist_track


## SQL Statements

Most actions you perform on a database are done with SQL statements. Note that SQL statements are **not case sensitive**.

### 1. SELECT
The `SELECT` statement is used to select data from a database. Using the `*` command gets all columns of the data.

In [106]:
run_query("SELECT album_id, title FROM album;")

,album_id,title
0,1,For Those About To Rock We Salute You
1,2,Balls to the Wall
2,3,Restless and Wild
3,4,Let There Be Rock
4,5,Big Ones
...,...,...
342,343,Respighi:Pines of Rome
343,344,Schubert: The Late String Quartets & String Qu...
344,345,Monteverdi: L'Orfeo
345,346,Mozart: Chamber Music


### 2. SELECT DISTINCT
The `SELECT DISTINCT` statement is used to return only distinct (different) values.

We can also combine this with `COUNT` to return the number of unique values and use `AS` to name the column something meaningful.

In [107]:
# Get distinct artist IDs
run_query("SELECT DISTINCT artist_id FROM album;")

,artist_id
0,1
1,2
2,3
3,4
4,5
...,...
199,271
200,272
201,273
202,274


In [108]:
# Count distinct artist IDs and rename column
run_query("SELECT COUNT(DISTINCT artist_id) as unique_artist_id FROM album;")

,unique_artist_id
0,204


### 3. LIMIT
The `LIMIT` clause is used to specify the number of records to return. This is useful on large tables with thousands of records, as returning a large number can impact performance.

In [109]:
run_query("""SELECT first_name, last_name
FROM customer
LIMIT 5""")

,first_name,last_name
0,Luís,Gonçalves
1,Leonie,Köhler
2,François,Tremblay
3,Bjørn,Hansen
4,František,Wichterlová


### 4. WHERE Clause
The `WHERE` clause is used to filter records that fulfill a specified condition.

**Operators:**
* **Standard:** `=`, `<`, `>`, `<=`
* **Logic:** `AND`, `OR`, `NOT`
* **Null Checks:** `IS NULL`, `IS NOT NULL` (to find missing values)

In [110]:
# Filtering by ID
run_query(
"""SELECT customer_id, first_name, last_name
FROM customer
WHERE customer_id = 5;""")

,customer_id,first_name,last_name
0,5,František,Wichterlová


In [111]:
# Using AND logic
run_query(
"""SELECT first_name, support_rep_id as representative
FROM customer
WHERE country = 'USA' AND representative = 3 """)

,first_name,representative
0,Michelle,3
1,Tim,3
2,Frank,3


In [112]:
# Checking for NULL (missing) Fax numbers
run_query(
"""SELECT first_name, last_name, fax
FROM customer
WHERE fax IS NULL LIMIT 5""")

,first_name,last_name,fax
0,Leonie,Köhler,None
1,François,Tremblay,None
2,Bjørn,Hansen,None
3,Helena,Holý,None
4,Astrid,Gruber,None


### 5. Advanced Filtering: LIKE and IN

* **LIKE:** Used to search for a specified pattern. The `%` sign represents zero, one, or multiple characters.
* **IN:** Allows you to specify multiple values in a WHERE clause.

In [113]:
# Pattern matching with LIKE
run_query(
"""SELECT first_name, last_name, country
FROM customer
WHERE country LIKE 'Netherlands' LIMIT 5""")

,first_name,last_name,country
0,Johannes,Van der Berg,Netherlands


In [114]:
# Multiple values with IN
run_query(
"""SELECT first_name, last_name, fax
FROM customer
WHERE country IN ('Netherlands', 'Germany') LIMIT 5""")

,first_name,last_name,fax
0,Leonie,Köhler,None
1,Hannah,Schneider,None
2,Fynn,Zimmermann,None
3,Niklas,Schröder,None
4,Johannes,Van der Berg,None


### 6. ORDER BY
The `ORDER BY` keyword sorts the result-set in ascending (default) or descending order[cite: 293, 311].
* Use `DESC` for descending order.
* For strings, it orders alphabetically.

In [115]:
# Order by representative (Ascending default)
run_query(
"""SELECT first_name, support_rep_id as representative
FROM customer
ORDER BY representative LIMIT 5""")

,first_name,representative
0,Luís,3
1,François,3
2,Roberto,3
3,Jennifer,3
4,Michelle,3


In [116]:

# Order by representative (Descending)
run_query(
"""SELECT first_name, support_rep_id as representative
FROM customer
ORDER BY representative DESC LIMIT 5""")

,first_name,representative
0,Luis,5
1,Steve,5
2,Joakim,5
3,Enrique,5
4,Johannes,5


In [117]:
# Alphabetical Order
run_query(
"""SELECT first_name, support_rep_id as representative
FROM customer
ORDER BY first_name LIMIT 5""")

,first_name,representative
0,Aaron,4
1,Alexandre,5
2,Astrid,5
3,Bjørn,4
4,Camille,4


### 7. Aggregate Functions & GROUP BY

SQL has functions to explore data elements, including `MIN`, `MAX`, `COUNT`, `SUM`, and `AVG`.

The `GROUP BY` statement groups rows that have the same values into summary rows (e.g., "find the number of tracks in each genre"). It is often used with aggregate functions.

In [118]:
# Using Aggregate Functions
run_query(
"""SELECT (MIN(milliseconds) / 60000) AS 'Shortest Track', 
       (MAX(milliseconds) / 60000) AS 'Longest Track',
       (AVG(milliseconds)  / 60000) AS 'Average Track'
FROM track
""")

,Shortest Track,Longest Track,Average Track
0,0,88,6.559987


In [119]:
# Using GROUP BY
run_query(
"""SELECT genre_id,
(AVG(milliseconds)  / 60000) AS 'Average Track'
FROM track
GROUP BY genre_id 
ORDER BY 'Average Track' DESC""")

,genre_id,Average Track
0,1,4.731834
1,2,4.862590
2,3,5.162491
3,4,3.905897
4,5,2.244058
5,6,4.505996
6,7,3.880988
7,8,4.119629
8,9,3.817235
9,10,4.072848


# Pandas
Pandas is the core library for data manipulation in Python. Its primary data structure is the **DataFrame**.

In [120]:
track_dict = {
    'TrackId': [1, 2, 3],
    'Name': ['For Those About To Rock', 'Balls to the Wall', 'Fast As a Shark'],
    'Composer': ['Angus Young', 'Accept', 'Accept'],
    'Milliseconds': [343719, 342562, 230619]
}

# Create DataFrame
df_dict = pd.DataFrame(track_dict)
df_dict

,TrackId,Name,Composer,Milliseconds
0,1,For Those About To Rock,Angus Young,343719
1,2,Balls to the Wall,Accept,342562
2,3,Fast As a Shark,Accept,230619


### Reading from CSV
In the real world, you will usually load data from external files like CSVs. The `pd.read_csv()` function is the standard tool for this.

**How to read correctly:**
* **Separators:** Standard CSVs use commas, but some use semicolons (`;`) or tabs (`\t`). You can fix this with `sep=';'`.
* **Headers:** By default, Pandas assumes the first row is the header. If your file has no header, use `header=None`.

In [121]:
df_csv = pd.read_csv('data/Track.csv', header=None)
df_csv.columns = ["TrackId","Name","AlbumId","MediaTypeId","GenreId","Composer","Milliseconds","Bytes","UnitPrice"]
df_csv.head()

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,NaN,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99
3,4,Restless and Wild,3,2,1,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",252051,4331779,0.99
4,5,Princess of the Dawn,3,2,1,Deaffy & R.A. Smith-Diesel,375418,6290521,0.99


## First stats about the data

We can get a basic overview of our data using `.info()` and `.desc()`

In [122]:
df_csv.info()

<class 'pandas.DataFrame'>
RangeIndex: 3503 entries, 0 to 3502
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   TrackId       3503 non-null   int64  
 1   Name          3503 non-null   str    
 2   AlbumId       3503 non-null   int64  
 3   MediaTypeId   3503 non-null   int64  
 4   GenreId       3503 non-null   int64  
 5   Composer      2525 non-null   str    
 6   Milliseconds  3503 non-null   int64  
 7   Bytes         3503 non-null   int64  
 8   UnitPrice     3503 non-null   float64
dtypes: float64(1), int64(6), str(2)
memory usage: 246.4 KB


In [123]:
df_csv.describe()

,TrackId,AlbumId,MediaTypeId,GenreId,Milliseconds,Bytes,UnitPrice
count,3503.000000,3503.000000,3503.000000,3503.000000,3.503000e+03,3.503000e+03,3503.000000
mean,1752.000000,140.929489,1.208393,5.725378,3.935992e+05,3.351021e+07,1.050805
std,1011.373324,81.775395,0.580443,6.190204,5.350054e+05,1.053925e+08,0.239006
min,1.000000,1.000000,1.000000,1.000000,1.071000e+03,3.874700e+04,0.990000
25%,876.500000,70.500000,1.000000,1.000000,2.072810e+05,6.342566e+06,0.990000
50%,1752.000000,141.000000,1.000000,3.000000,2.556340e+05,8.107896e+06,0.990000
75%,2627.500000,212.000000,1.000000,7.000000,3.216450e+05,1.026679e+07,0.990000
max,3503.000000,347.000000,5.000000,25.000000,5.286953e+06,1.059546e+09,1.990000


## Navigating the DataFrame

You can easily isolate specific parts of your data, either by selecting specific columns (features) or rows (records).

### 1. Selecting Columns
To select a column, we use square brackets `[]` with the column name string inside.

In [124]:
# Select a single column (returns a Series)
names = df_csv[['Name']]
names.head()

,Name
0,For Those About To Rock (We Salute You)
1,Balls to the Wall
2,Fast As a Shark
3,Restless and Wild
4,Princess of the Dawn


In [125]:
# Select multiple columns (returns a DataFrame)
subset = df_csv[['Name', 'Composer', 'Milliseconds']]
subset.head()

,Name,Composer,Milliseconds
0,For Those About To Rock (We Salute You),"Angus Young, Malcolm Young, Brian Johnson",343719
1,Balls to the Wall,NaN,342562
2,Fast As a Shark,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619
3,Restless and Wild,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",252051
4,Princess of the Dawn,Deaffy & R.A. Smith-Diesel,375418


### 2. Selecting Rows (Indexing)

Pandas uses two primary methods for selecting rows:
* `.iloc`: Select by **integer position** (like lists, e.g., row 0 is the first row).
* `.loc`: Select by **label** (index name). 

Since our dataframe currently has a default integer index, `.loc` and `.iloc` will appear similar, but the concept  is distinctly different when using custom indices.

In [126]:
# Select the first row by position (iloc)
print("--- First Row ---")
print(df_csv.iloc[0])

# Select rows 5 to 10 (slicing)
print("\n--- Rows 5 to 10 ---")
df_csv.iloc[5:10]

--- First Row ---
TrackId                                                 1
Name              For Those About To Rock (We Salute You)
AlbumId                                                 1
MediaTypeId                                             1
GenreId                                                 1
Composer        Angus Young, Malcolm Young, Brian Johnson
Milliseconds                                       343719
Bytes                                            11170334
UnitPrice                                            0.99
Name: 0, dtype: object

--- Rows 5 to 10 ---


,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
5,6,Put The Finger On You,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",205662,6713451,0.99
6,7,Let's Get It Up,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",233926,7636561,0.99
7,8,Inject The Venom,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",210834,6852860,0.99
8,9,Snowballed,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",203102,6599424,0.99
9,10,Evil Walks,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",263497,8611245,0.99


In [127]:
df_csv.loc[5:10, ['Name']]

,Name
5,Put The Finger On You
6,Let's Get It Up
7,Inject The Venom
8,Snowballed
9,Evil Walks
10,C.O.D.


### 3. Filtering

We can filter pandas tables as well.

In [128]:
df_csv.loc[df_csv['Composer'] == 'Franz Schubert', ['Name']]


,Name
3415,Ave Maria
3499,"String Quartet No. 12 in C Minor, D. 703 ""Quar..."


We can store the filter separately and use it afterwards on top of our dataframe

In [129]:
mask = df_csv['Composer'] == 'Franz Schubert'
df_csv.loc[mask, ['Name']]

,Name
3415,Ave Maria
3499,"String Quartet No. 12 in C Minor, D. 703 ""Quar..."


Our mask can also contain boolean operators like and (`&`), or (`|`) and not (`~`)

In [130]:
mask = (df_csv['Milliseconds'] > 1000) & (df_csv['UnitPrice'] > 0.99)
df_csv.loc[mask, ['Name']]

,Name
2818,Battlestar Galactica: The Story So Far
2819,Occupation / Precipice
2820,"Exodus, Pt. 1"
2821,"Exodus, Pt. 2"
2822,Collaborators
...,...
3361,"There's No Place Like Home, Pt. 1"
3362,"There's No Place Like Home, Pt. 2"
3363,"There's No Place Like Home, Pt. 3"
3427,Branch Closing
